In [ ]:
from io import BytesIO
import math
from pathlib import Path
from urllib.parse import quote

import ipywidgets as widgets

from IPython.display import display
from matplotlib import pyplot

from searchlight.lightcurves import load, plot
from searchlight.models.simulens.v1 import (
    DEFAULT_SPLIT_SIZES,
    _segment_lightcurve,
    _segment_windows_for_split_size,
)
from searchlight.review_exports import (
    RESULT_COLUMNS,
    RESULT_DTYPES,
    DEFAULT_METADATA_CACHE_PATH,
    DEFAULT_RESULT_CACHE_META_PATH,
    DEFAULT_RESULT_CACHE_PATH,
    enrich_results_with_metadata,
    load_result_frames,
)

In [ ]:
RESULTS_PATH = Path('data/results/simulens1.0_tess_spoc')
SUPPRESSION_PATH = RESULTS_PATH / 'top_resnet_suppressions.txt'
RESULT_CACHE_PATH = DEFAULT_RESULT_CACHE_PATH
RESULT_CACHE_META_PATH = DEFAULT_RESULT_CACHE_META_PATH
RESULT_EXPORT_PATH = RESULTS_PATH / 'top_resnet_final_results.csv'
RESULT_METADATA_CACHE_PATH = DEFAULT_METADATA_CACHE_PATH


def load_suppression_list():
    if not SUPPRESSION_PATH.exists():
        return set()
    return {
        line.strip()
        for line in SUPPRESSION_PATH.read_text().splitlines()
        if line.strip()
    }


def add_to_suppression_list(path):
    suppressed = load_suppression_list()
    if path in suppressed:
        return False
    SUPPRESSION_PATH.parent.mkdir(parents=True, exist_ok=True)
    with SUPPRESSION_PATH.open('a') as handle:
        handle.write(f'{path}\n')
    return True


def export_final_results(results):
    export = enrich_results_with_metadata(results, cache_path=RESULT_METADATA_CACHE_PATH)
    export = export[['path', 'lensing', 'tic', 'sector', 'simbad_url']].copy()
    export = export.rename(columns={'lensing': 'score'})
    export = export.rename(columns={'tic': 'TIC', 'simbad_url': 'SIMBAD'})
    export.to_csv(RESULT_EXPORT_PATH, index=False)
    print(f'Wrote {len(export)} rows to {RESULT_EXPORT_PATH}')
    return export


def make_review_item(path, score, lc):
    button = widgets.Button(
        description='Suppress',
        button_style='warning',
        tooltip='Add this path to the persisted suppression list',
    )
    status = widgets.HTML()
    header = widgets.HBox([
        button,
        widgets.HTML(f'<code>{Path(path).name}</code>'),
        status,
    ])

    ticid = lc.meta.get('TICID') or lc.meta.get('TARGETID')
    sector = lc.meta.get('SECTOR')
    ra = lc.meta.get('RA')
    dec = lc.meta.get('DEC')

    lines = [
        f'<div><strong>File:</strong> <code>{path}</code></div>',
        f'<div><strong>Score:</strong> {score}</div>',
    ]
    if ticid is not None:
        lines.append(f'<div><strong>TIC:</strong> {ticid}</div>')
    if sector is not None:
        lines.append(f'<div><strong>Sector:</strong> {sector}</div>')
    if ra is not None and dec is not None:
        coord = f'{ra} {dec}'
        simbad_url = 'https://simbad.cds.unistra.fr/simbad/sim-coo?Coord=' + quote(coord)
        lines.append(f'<div><strong>SIMBAD:</strong> <a href="{simbad_url}" target="_blank">{simbad_url}</a></div>')
    details = widgets.HTML(''.join(lines))

    split_children = []
    open_index = None

    def split_label(split_size_days):
        if math.isinf(split_size_days):
            return 'all data'
        return f'{split_size_days:g} days'

    for split_size in DEFAULT_SPLIT_SIZES:
        split_size_days = float(split_size)
        windows = _segment_windows_for_split_size(lc, split_size_days)
        segments = [
            _segment_lightcurve(lc, start, end)
            for start, end, _ in windows
        ]
        segments = [seg for seg in segments if len(seg) > 0]

        if not segments:
            split_children.append(widgets.HTML(f'<div>No non-empty segments for <code>{split_label(split_size_days)}</code>.</div>'))
            if math.isinf(split_size_days):
                open_index = len(split_children) - 1
            continue

        fig, axes = pyplot.subplots(
            len(segments),
            1,
            figsize=(12, max(2.5, 2.5 * len(segments))),
            squeeze=False,
        )
        axes = axes.flatten()
        fig.suptitle(f'{Path(path).name} | split_size={split_size_days} days', fontsize=12)

        for i, seg in enumerate(segments):
            ax = axes[i]
            ax.scatter(seg['time'].jd, seg['flux'], s=2, alpha=0.6)
            ax.set_ylabel('flux')
            ax.set_title(f'segment {i + 1} ({len(seg)} points)', fontsize=10)

        axes[-1].set_xlabel('time (JD)')
        pyplot.tight_layout()

        buffer = BytesIO()
        fig.savefig(buffer, format='png', bbox_inches='tight')
        pyplot.close(fig)
        split_children.append(widgets.Image(value=buffer.getvalue(), format='png'))
        if math.isinf(split_size_days):
            open_index = len(split_children) - 1

    plots = widgets.Accordion(children=split_children)
    for index, split_size in enumerate(DEFAULT_SPLIT_SIZES):
        plots.set_title(index, split_label(float(split_size)))
    plots.selected_index = open_index

    content = widgets.VBox([details, plots])
    card = widgets.VBox([header, content])
    card.layout.border = '1px solid #ddd'
    card.layout.padding = '8px'
    card.layout.margin = '0 0 16px 0'

    def mark_suppressed(message):
        button.disabled = True
        button.description = 'Suppressed'
        button.button_style = ''
        status.value = f'<span style="color:#666;font-style:italic;">{message}</span>'
        content.layout.display = 'none'

    if path in load_suppression_list():
        mark_suppressed('Already suppressed')

    def on_click(_):
        added = add_to_suppression_list(path)
        if added:
            mark_suppressed(f'Saved to {SUPPRESSION_PATH}')
        else:
            mark_suppressed('Already in suppression list')

    button.on_click(on_click)
    return card


In [ ]:
result_paths = list(RESULTS_PATH.glob('*_resnet.csv'))
results = load_result_frames(result_paths, RESULT_COLUMNS, RESULT_DTYPES)
suppressed_paths = load_suppression_list()
suppressed_matches = results['path'].isin(suppressed_paths)
results = results[~suppressed_matches].copy()
print(f'Suppression file: {SUPPRESSION_PATH}')
print(f'Suppressed matches removed: {suppressed_matches.sum()}')

In [ ]:
results['label'].value_counts()

In [ ]:
results = results[results['label'] == 'lensing']

In [ ]:
results['lensing'].plot.hist()

In [ ]:
results = results[results['lensing'] > 2]
len(results)

In [ ]:
results.sort_values('lensing', inplace=True, ascending=False)

In [ ]:
results = results.head(100)

In [ ]:
final_results_export = export_final_results(results)
final_results_export.head()

In [ ]:
for path, score in results[['path', 'lensing']].values:
    lc = load(path)
    display(make_review_item(path, score, lc))